# Heart Disease Prediction: step-by-step
Runs the same pipeline as `run_pipeline.py`, one stage per cell. Run from the `notebooks/` folder after placing `data/heart.csv` (see `data/README.md`).

In [1]:
import sys
sys.path.insert(0, "..")   # make the heart_ml package importable
import pandas as pd
import matplotlib.pyplot as plt
from heart_ml.config import DEFAULT_DATA_PATH, MODEL_LABELS
from heart_ml.data import load_dataset, dataset_summary, format_summary
from heart_ml.preprocess import build_preprocessor, split_data
from heart_ml.models import get_models
from heart_ml.evaluate import evaluate_on_test, results_table, select_best, cross_validate_all
from heart_ml import plots

## 1. Load and inspect

In [2]:
df = load_dataset(DEFAULT_DATA_PATH)
print(format_summary(dataset_summary(df)))
df.head()

Rows x columns : 303 x 14
Missing values : 6 {'ca': 4, 'thal': 2}
Duplicate rows : 0
Class balance  : {0: 164, 1: 139} (positive rate 45.9%)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## 2. Exploratory data analysis

In [4]:
from pathlib import Path
paths = plots.plot_eda(df, Path("../results/eda"))
for p in paths:
    plt.figure(figsize=(9, 5)); plt.imshow(plt.imread(p)); plt.axis("off")

## 3. Preprocess and split
The transformer is fitted on the training set only.

In [5]:
SEED = 42
X_train, X_test, y_train, y_test = split_data(df, seed=SEED)
prep = build_preprocessor()
Xtr = prep.fit_transform(X_train)
Xte = prep.transform(X_test)
print(len(X_train), "train /", len(X_test), "test;", Xtr.shape[1], "features")
Xtr.head()

242 train / 61 test; 18 features


,age,trestbps,chol,thalach,oldpeak,cp_2.0,cp_3.0,cp_4.0,restecg_1.0,restecg_2.0,slope_2.0,slope_3.0,thal_6.0,thal_7.0,sex,fbs,exang,ca
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


## 4. Train all eight models on the same split

In [6]:
models = get_models(seed=SEED)      # add include_ann=False to skip Keras
results = []
for key, model in models.items():
    model.fit(Xtr, y_train)
    results.append(evaluate_on_test(key, model, Xte, y_test))
table = results_table(results)
best = select_best(results)
table.assign(**{c: (table[c]*100).round(1) for c in ["accuracy","precision","recall","f1"]})

[warn] xgboost not installed - skipping XGBoost (pip install xgboost)


,key,model,accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp
0,LR,Logistic Regression,85.2,80.6,89.3,84.7,0.956710,27,6,3,25
1,NB,Naive Bayes,60.7,54.3,89.3,67.6,0.847403,12,21,3,25
2,SVM,Linear SVM,88.5,83.9,92.9,88.1,0.954545,28,5,2,26
3,KNN,K-Nearest Neighbours,91.8,87.1,96.4,91.5,0.966450,29,4,1,27
4,DT,Decision Tree,80.3,76.7,82.1,79.3,0.804654,26,7,5,23
5,RF,Random Forest,90.2,84.4,96.4,90.0,0.956710,28,5,1,27
6,ANN,ANN (1 hidden layer),90.2,86.7,92.9,89.7,0.965368,29,4,2,26


## 5. Compare

In [7]:
plots.plot_accuracy_comparison(table, best["key"], Path("../results/accuracy_comparison.png"))
plt.figure(figsize=(9, 4.5)); plt.imshow(plt.imread("../results/accuracy_comparison.png")); plt.axis("off")
print("Best:", MODEL_LABELS[best["key"]]); print(best["report"])

Best: K-Nearest Neighbours
                precision    recall  f1-score   support

No disease (0)       0.97      0.88      0.92        33
   Disease (1)       0.87      0.96      0.92        28

      accuracy                           0.92        61
     macro avg       0.92      0.92      0.92        61
  weighted avg       0.92      0.92      0.92        61



In [8]:
plots.plot_confusion_matrix(best["confusion_matrix"], MODEL_LABELS[best["key"]] + ": confusion matrix",
                            Path("../results/confusion_matrix_best.png"))
plt.figure(figsize=(4.4, 3.8)); plt.imshow(plt.imread("../results/confusion_matrix_best.png")); plt.axis("off")

(np.float64(-0.5), np.float64(703.5), np.float64(607.5), np.float64(-0.5))

## 6. More stable estimate: repeated cross-validation
A single 61-patient test split is noisy (see the report's limitations). Preprocessing is refit inside every fold.

In [9]:
cv = cross_validate_all(models, df, n_splits=5, n_repeats=5, seed=SEED, ann_repeats=1)
cv[["model", "accuracy_mean", "accuracy_std", "recall_mean", "f1_mean"]].round(3)

,model,accuracy_mean,accuracy_std,recall_mean,f1_mean
0,Logistic Regression,0.840,0.045,0.783,0.817
1,Naive Bayes,0.793,0.070,0.710,0.747
2,Linear SVM,0.838,0.046,0.781,0.815
3,K-Nearest Neighbours,0.828,0.043,0.787,0.806
4,Decision Tree,0.750,0.053,0.745,0.731
5,Random Forest,0.829,0.050,0.771,0.804
6,ANN (1 hidden layer),0.812,0.042,0.770,0.791
